# Simulation Study

Online balancing.

In this notebook, we demonstrate the efficacy of our online discrepancy minimization algorithm in comparison to:

1. Complete randomization
2. Naïve Alweiss (2020) balancing
3. Offline balancing via QuickBlock
4. Efron's biased coin design
5. Simple (bernoulli) randomization
6. Simple randomization with Lin-style covariate adjustment

In [1]:
# !pip3 install --upgrade -r requirements.txt
!julia -e 'using Pkg; Pkg.add("https://github.com/crharshaw/GSWDesign.jl")'

zsh:1: command not found: julia


In [37]:
!conda install --yes -c conda-forge Julia
!julia -v

done
Solving environment: | ^C
failed with initial frozen solve. Retrying with flexible solve.

CondaError: KeyboardInterrupt

zsh:1: command not found: julia


In [2]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

In [3]:
import os.path

import numpy as np
from tqdm import tqdm

In [4]:
from alias import *
from utils import *

In [5]:
plan = make_plan(
    [
        # ('Complete', bal.Complete, est.DifferenceInMeans, {}),
        ("Simple", bal.Simple, est.DifferenceInMeans, {}),
        # ('Simple + OLS', bal.Simple, est.CovariateAdjustedMean, {}),
        ("Efron", bal.Efron, est.DifferenceInMeans, {}),
        ("Alweiss", bal.Alweiss, est.DifferenceInMeans, {}),
        ("QuickBlock", bal.QuickBlock, est.DifferenceInMeans, {}),
        # ('QuickBlock + OLS', bal.QuickBlock, est.CovariateAdjustedMean, {}),
        ("BWD(1)", bal.BWD, est.DifferenceInMeans, {"phi": 1}),
        ("BWD(0.5)", bal.BWD, est.DifferenceInMeans, {"phi": 0.5}),
        # ('BWD + OLS', bal.BWD, est.CovariateAdjustedMean, {}),
        ("BWDRandom", bal.BWDRandom, est.DifferenceInMeans, {}),
        # ('BWDRandom + OLS', bal.BWDRandom, est.CovariateAdjustedMean, {}),
        ("DM", bal.DM, est.DifferenceInMeans, {}),
        # ('DM + OLS', bal.DM, est.CovariateAdjustedMean, {}),
    ]
)

In [7]:
import os

os.makedirs("main-results", exist_ok=True)

In [9]:
dfs = []
dgp_factory_class_list = [
    dgp.LinearFactory,
    dgp.QuickBlockFactory,
    dgp.LinearDriftFactory,
    dgp.LinearSeasonFactory,
    dgp.QuadraticFactory,
    dgp.CubicFactory,
    dgp.SinusoidalFactory,
    # dgp.LinearFactory,dgp.LinearDriftFactory,dgp.LinearSeasonFactory,dgp.QuickBlockFactory
    # dgp.LinearSeasonFactory
]
sample_sizes = np.floor(
    np.logspace(np.log10(100), np.log10(10_000), base=10, num=12)
).astype(int)
# sample_sizes = np.floor(np.logspace(np.log10(100), np.log10(1_000), base=10, num=6)).astype(int)

NUM_ITERS = 10

for sample_size in sample_sizes[::-1]:
    print(f"Sample Size: {sample_size}", flush=True)
    dgp_factory_list = [
        factory(N=sample_size) for factory in dgp_factory_class_list[0:]
    ]
    for dgp_factory in dgp_factory_list:
        dgp_name = type(dgp_factory.create_dgp()).__name__
        print(f"\nDGP name: {dgp_name}", flush=True)
        for it in tqdm(range(NUM_ITERS)):
            filename = f"main-results/{dgp_name}_n{sample_size}_i{it}.csv.gz"
            if not os.path.isfile(filename):
                result = plan.execute(dgp_factory, seed=it * 1001)
                result["iteration"] = it
                result["sample_size"] = sample_size
                result["dgp"] = dgp_name
                result.to_csv(filename, index=False)
                # upload_file(filename, BUCKET)
                dfs.append(result)

Sample Size: 10000

DGP name: LinearDGP


100%|██████████| 10/10 [00:00<00:00, 6394.73it/s]


DGP name: QuickBlockDGP



100%|██████████| 10/10 [00:00<00:00, 22610.80it/s]


DGP name: LinearDriftDGP


...


100%|██████████| 10/10 [00:00<00:00, 9578.22it/s]


DGP name: CubicDGP



100%|██████████| 10/10 [00:00<00:00, 32896.50it/s]


DGP name: SinusoidalDGP



100%|██████████| 10/10 [00:00<00:00, 31138.11it/s]


In [11]:
plan = make_plan(
    [
        # ('Complete', bal.Complete, est.DifferenceInMeans, {}),
        ("Simple", bal.Simple, est.DifferenceInMeans, {}),
        # ('Simple + OLS', bal.Simple, est.CovariateAdjustedMean, {}),
        ("Efron", bal.Efron, est.DifferenceInMeans, {}),
        ("Alweiss", bal.Alweiss, est.DifferenceInMeans, {}),
        # ('QuickBlock + OLS', bal.QuickBlock, est.CovariateAdjustedMean, {}),
        ("BWD(1)", bal.BWD, est.DifferenceInMeans, {"phi": 1}),
        ("BWD(0.5)", bal.BWD, est.DifferenceInMeans, {"phi": 0.5}),
        # ('BWD + OLS', bal.BWD, est.CovariateAdjustedMean, {}),
        ("BWDRandom", bal.BWDRandom, est.DifferenceInMeans, {}),
        # ('BWDRandom + OLS', bal.BWDRandom, est.CovariateAdjustedMean, {}),
        ("DM", bal.DM, est.DifferenceInMeans, {}),
        # ('DM + OLS', bal.DM, est.CovariateAdjustedMean, {}),
    ]
)
# plan = make_plan([
#     ('Simple', bal.Simple, est.DifferenceInMeans, {}),
#     #('Simple + OLS', bal.Simple, est.CovariateAdjustedMean, {}),
#     ('Efron', bal.Efron, est.DifferenceInMeans, {}),
#     ('Alweiss', bal.Alweiss, est.DifferenceInMeans, {}),
#     ('BWD', bal.BWD, est.DifferenceInMeans, {}),
#     #('BWD + OLS', bal.BWD, est.CovariateAdjustedMean, {}),
#     ('DM', bal.DM, est.DifferenceInMeans, {}),
#     #('DM + OLS', bal.DM, est.CovariateAdjustedMean, {}),
# ])
# sample_sizes = [25_000, 50_000, 100_000]
sample_sizes = np.floor(
    np.logspace(np.log10(10_000), np.log10(100_000), base=10, num=8)
).astype(int)

NUM_ITERS = 10

for sample_size in sample_sizes:
    print(f"Sample Size: {sample_size}", flush=True)
    dgp_factory_list = [factory(N=sample_size) for factory in dgp_factory_class_list]
    for dgp_factory in dgp_factory_list:
        dgp_name = type(dgp_factory.create_dgp()).__name__
        print(f"\nDGP name: {dgp_name}", flush=True)
        for it in tqdm(range(NUM_ITERS)):
            filename = f"main-results/{dgp_name}_n{sample_size}_i{it}.csv.gz"
            if not os.path.isfile(filename):
                result = plan.execute(dgp_factory, seed=it * 1001)
                result["iteration"] = it
                result["sample_size"] = sample_size
                result["dgp"] = dgp_name
                result.to_csv(filename, index=False)
                # upload_file(filename, BUCKET)
                dfs.append(result)

Sample Size: 10000

DGP name: LinearDGP


100%|██████████| 10/10 [00:00<00:00, 5571.60it/s]


DGP name: QuickBlockDGP



100%|██████████| 10/10 [00:00<00:00, 8156.95it/s]


DGP name: LinearDriftDGP


...


100%|██████████| 10/10 [00:21<00:00,  2.18s/it]


DGP name: CubicDGP



100%|██████████| 10/10 [00:21<00:00,  2.13s/it]


DGP name: SinusoidalDGP



100%|██████████| 10/10 [00:20<00:00,  2.07s/it]
